In [1]:
%run std_methods.ipynb

In [2]:
import pandas as pd, math, datetime

In [3]:
logger =  initialize_logging()

In [4]:
def generate_date_sequence(start_date_YYYYMMDD,end_date_YYYYMMDD,period):
    
    period = period.lower()
    start_date = datetime.datetime.strptime(start_date_YYYYMMDD,'%Y-%m-%d')
    end_date = datetime.datetime.strptime(end_date_YYYYMMDD,'%Y-%m-%d')
    
    if period == "monthly":
        first_of_starting_month=start_date.strftime('%Y-%m-')+"01"
        date_offset=start_date-datetime.datetime.strptime(first_of_starting_month,'%Y-%m-%d')
        relevant_month_starts =  pd.date_range(start=first_of_starting_month, end=end_date, freq='MS').to_pydatetime().tolist()
        return [ dt + date_offset for dt in relevant_month_starts ]

    if period == "daily":
        return pd.date_range(start=start_date, end=end_date, freq='D').to_pydatetime().tolist()

    if period == "90 days":
        number_of_90_day_periods = math.floor((end_date - start_date).days / 90)
        return [start_date + datetime.timedelta(days=offset_index*90) for offset_index in range(0, number_of_90_day_periods)]

    if period == "biweekly":
        number_of_14_day_periods = math.floor((end_date - start_date).days / 14)
        return [start_date + datetime.timedelta(days=offset_index*14) for offset_index in range(0, number_of_14_day_periods)]

def generate_series_from_item_schedule(budget_data,accounts_df,start_date_YYYYMMDD,num_days):

    start_date = datetime.datetime.strptime(start_date_YYYYMMDD,'%Y-%m-%d')
    end_date = start_date + datetime.timedelta(days=num_days)
    end_date_YYYYMMDD = end_date.strftime('%Y-%m-%d')
    
    account_name_to_type_map = get_account_name_to_type_map(accounts_df)

    #Monthly
    monthly_items_df=budget_data.loc[budget_data['Cadence'] == "Monthly"]
    monthly_items_list=[]
    for index, row in monthly_items_df.iterrows():
        current_start_date_YYYYMMDD = row["Start Date"]
        current_amount = row["Amount"]
        current_paid_from = row["Paid From"]
        current_paid_to = row["Paid To"]
        current_priority = row["Priority"]
        current_memo = row["Memo"]

        current_monthly_sequence = generate_date_sequence(current_start_date_YYYYMMDD,end_date_YYYYMMDD,"monthly")
    
        for dt in current_monthly_sequence:
            monthly_items_list.append([dt,current_amount,current_paid_from,current_paid_to,current_priority,current_memo])
            

    #Daily
    daily_items_df=budget_data.loc[budget_data['Cadence'] == "Daily"]
    daily_items_list=[]
    for index, row in daily_items_df.iterrows():
        current_start_date_YYYYMMDD = row["Start Date"]
        current_amount = row["Amount"]
        current_paid_from = row["Paid From"]
        current_paid_to = row["Paid To"]
        current_priority = row["Priority"]
        current_memo = row["Memo"]
        
        current_daily_sequence = generate_date_sequence(current_start_date_YYYYMMDD, end_date_YYYYMMDD, "daily")
 
        for dt in current_daily_sequence:            
            daily_items_list.append([dt,current_amount,current_paid_from,current_paid_to,current_priority,current_memo])
                    

    #Bi-weekly
    biweekly_items_df = budget_data.loc[budget_data['Cadence'] == "Biweekly"]
    biweekly_items_list = []
    for index, row in biweekly_items_df.iterrows():
        current_start_date_YYYYMMDD = row["Start Date"]
        current_amount = row["Amount"]
        current_paid_from = row["Paid From"]
        current_paid_to = row["Paid To"]
        current_priority = row["Priority"]
        current_memo = row["Memo"]

        current_biweekly_sequence = generate_date_sequence(current_start_date_YYYYMMDD, end_date_YYYYMMDD, "biweekly")
  
        for dt in current_biweekly_sequence:
            biweekly_items_list.append([dt,current_amount,current_paid_from,current_paid_to,current_priority,current_memo])
                    

    #Once
    once_items_df=budget_data.loc[budget_data['Cadence'] == "Once"]
    once_items_list=[]
    for index, row in once_items_df.iterrows():
        current_start_date_YYYYMMDD = row["Start Date"]
        current_amount = row["Amount"]
        current_paid_from = row["Paid From"]
        current_paid_to = row["Paid To"]
        current_priority = row["Priority"]
        current_memo = row["Memo"]
  
        once_items_list.append([dt,current_amount,current_paid_from,current_paid_to,current_priority,current_memo])
                

    #Every 3 months
    every_3_months_items_df=budget_data.loc[budget_data['Cadence'] == "Every 3 months"]
    every_3_months_list=[]
    for index, row in every_3_months_items_df.iterrows():
        current_start_date_YYYYMMDD = row["Start Date"]
        current_amount = row["Amount"]
        current_paid_from = row["Paid From"]
        current_paid_to = row["Paid To"]
        current_priority = row["Priority"]
        current_memo = row["Memo"]

        current_90_day_sequence = generate_date_sequence(current_start_date_YYYYMMDD, end_date_YYYYMMDD, "90 days")
        
        for dt in current_90_day_sequence:
            every_3_months_list.append([dt,current_amount,current_paid_from,current_paid_to,current_priority,current_memo])

    scheduled_items_df = pd.DataFrame(monthly_items_list + daily_items_list + every_3_months_list + once_items_list + biweekly_items_list)
    scheduled_items_df.columns = ["Date","Amount","Account From","Account To","Priority","Memo"]
    scheduled_items_df = scheduled_items_df[scheduled_items_df.Date >= start_date]
    return scheduled_items_df.sort_values(by="Date")

def validate_output(output_df):
    
    output_df.to_csv('/home/hdickie/sandbox/output.csv')
    
    data_source_name = "output"
    data_source_path = '/home/hdickie/sandbox/output.csv'
    data_asset_name="output"
    expectation_suite_name="Output_Validation"
    checkpoint_name="chkpt1"
    context, validator = getValidatorAndUpdatedContext(data_source_name, data_source_path, data_asset_name, expectation_suite_name, checkpoint_name)
    
    error_flag = False
    
    #assert table schema
    validator.expect_column_to_exist('Date')
    validator.expect_column_to_exist('Memo')
    error_flag = error_flag and not validator.expect_column_to_exist('Date').success #these method calls have side effects 
    error_flag = error_flag and not validator.expect_column_to_exist('Memo').success
    
    #assert Dt is unique
    validator.expect_column_values_to_be_unique('Date')
    error_flag = error_flag and not validator.expect_column_values_to_be_unique('Date').success
    
    if error_flag:
        logger.info("Errors were detected. Please check validation docs at file:///home/hdickie/great_expectations/uncommitted/data_docs/local_site/index.html")
    
    runValidation(context,validator,checkpoint_name,output_df)
    
    return error_flag

def validate_parameter_tables(budget_items_df, 
                              priority_account_rules_df, 
                              priority_item_rules_df, 
                              accounts_df, 
                              milestones_df):
    
    #Since i dont know how to use great ewxpectations all the way, this data type casting will not be reflected in great expectations tests
    budget_items_df['Start Date'] = pd.to_datetime(budget_items_df['Start Date'], format='%Y-%m-%d')
    
    
    #save dfs separately for great_expectations
    budget_items_df.to_csv('/home/hdickie/sandbox/budget_items.csv')
    priority_account_rules_df.to_csv('/home/hdickie/sandbox/priority_account_rules.csv')
    priority_item_rules_df.to_csv('/home/hdickie/sandbox/priority_item_rules.csv')
    accounts_df.to_csv('/home/hdickie/sandbox/accounts.csv')
    milestones_df.to_csv('/home/hdickie/sandbox/milestones.csv')
    
    error_flag = False
    
    ###By sheet:
    
    # Budget Items
    data_source_name = "expense_forecast_budget_items"
    data_source_path = '/home/hdickie/sandbox/budget_items.csv'
    data_asset_name="expense_forecast_budget_items"
    expectation_suite_name="expense_forecast"
    checkpoint_name="validate_parameter_tables"
    context, validator = getValidatorAndUpdatedContext(data_source_name, data_source_path, data_asset_name, expectation_suite_name, checkpoint_name)
    
    #assert table schema
    error_flag = error_flag and not validator.expect_column_to_exist('Start Date').success #these method calls have side effects 
    error_flag = error_flag and not validator.expect_column_to_exist('Priority').success
    error_flag = error_flag and not validator.expect_column_to_exist('Cadence').success
    error_flag = error_flag and not validator.expect_column_to_exist('Amount').success
    error_flag = error_flag and not validator.expect_column_to_exist('Paid From').success
    error_flag = error_flag and not validator.expect_column_to_exist('Paid To').success
    error_flag = error_flag and not validator.expect_column_to_exist('Memo').success
    
    #assert non-nulls
    error_flag = error_flag and not validator.expect_column_values_to_not_be_null('Priority').success
    error_flag = error_flag and not validator.expect_column_values_to_not_be_null('Cadence').success
    error_flag = error_flag and not validator.expect_column_values_to_not_be_null('Amount').success
    
    # assert that all start dates are valid
    error_flag = error_flag and not validator.expect_column_values_to_match_regex('Start Date','[0-9]{4}-(0[1-9]|1[0-2])-(0[1-9]|[12][0-9]|3[01])').success
    
    # assert that all priorities have defined behavior
    # TODO: if new priority levels become athign this will need to be adjusted
    error_flag = error_flag and not validator.expect_column_values_to_be_in_set('Cadence',[1,2,3,4,5,6]).success
    
    # assert that cadence values are in set
    error_flag = error_flag and not validator.expect_column_values_to_be_in_set('Cadence',['once','daily','weekly','monthly','biweekly','yearly','Once','Daily','Weekly','Monthly','Biweekly','Yearly']).success
    
    # assert that amounts are valid floats
    error_flag = error_flag and not validator.expect_column_values_to_be_of_type('Amount','float').success
    
    # assert that paid from/to values are in set
    #TODO: New accounts will need to be added here, possibly dynamically
    error_flag = error_flag and not validator.expect_column_values_to_be_in_set('Paid From',['Checking','Credit 1','Savings']).success
    error_flag = error_flag and not validator.expect_column_values_to_be_in_set('Paid To',['Checking','Credit 1','Savings']).success
    
    #runValidation(context,validator,checkpoint_name,budget_items_df)
    
    # Priority Account Rules
    #assert that all priority indices are integers (this is the source of truth column)
    #assert that account names are in set
    # assert Balance LB/UB are valid floats or +/-Inf
    
    # Priority Item Rules
    #would only need to validate if we were defining new priroity levels which is otuside of scope 
    
    # Accounts
    #assert balances columns are valid floats or +/-Inf
    #assert interest rate is valid float
    #assert interest type is Simple or Compound
    #assert billing_start_dt is valid dt
    
    # Milestones
    #assert account names are in set
    #assert min/max balacne are valid floats or +/-Inf
    
    if error_flag:
        logger.info("Errors were detected. Please check validation docs at file:///home/hdickie/great_expectations/uncommitted/data_docs/local_site/index.html")
    
    return error_flag

def read_budget_item_excel_sheet(excel_path,validate=False):
    budget_items_df = pd.read_excel(excel_path,sheet_name = 'Budget_Items')
    priority_account_rules_df = pd.read_excel(excel_path,sheet_name = 'Priority_Account_Rules')
    priority_item_rules_df = pd.read_excel(excel_path,sheet_name = 'Priority_Item_Rules')
    accounts_df = pd.read_excel(excel_path,sheet_name = 'Accounts')
    milestones_df = pd.read_excel(excel_path,sheet_name = 'Milestones')
    
    if validate:
        validate_parameter_tables(budget_items_df, priority_account_rules_df, priority_item_rules_df, accounts_df, milestones_df)
    
    return [budget_items_df, priority_account_rules_df, priority_item_rules_df, accounts_df, milestones_df]
    
def write_results(results):
    results.to_csv('/home/hdickie/sandbox/results.csv',index=False)
    
def get_initial_balances(accounts_df):
    acct_balances = {}
    for index,row in accounts_df.iterrows():
        acct_balances[row['Account Name']] = row['Current Balance']
    return acct_balances

def get_account_name_to_type_map(accounts_df):
    acct_types = {}
    for index,row in accounts_df.iterrows():
        acct_types[row['Account Name']] = row['Account Type']
    return acct_types


In [5]:
def reformat_acct_balance_row_to_log_string(today_df,header=True):
    assert today_df.shape[0] == 1 #only a single row should be input
    
    input_parameters_log_string = "\n"+"Date".ljust(11)+"|"
    
    ljust_values=[]
    for x in today_df.columns[1:len(today_df.columns)-1]:
        x = x.replace("Interest","Intrst")
        x = x.replace("Balance","Bal")
        x = x.replace("Checking","Chk")
        x = x.replace("Savings","Sav")
        x = x.replace("Grace Period","GP")
        x = x.replace("Credit","Crdt")
        x = x.replace("Loan","Ln")
        ljust_values += [max(len(x),8)]
        if header:
            input_parameters_log_string += str(x).ljust(max(len(x),8)) + "|"
    
    if header:
        input_parameters_log_string += "\n"
    input_parameters_log_string += today_df["Date"].iat[0].strftime('%Y-%m-%d').ljust(11)+"|"
    
    ljust_index = 0
    for x in today_df.columns[1:len(today_df.columns)-1]:
        input_parameters_log_string += str(today_df[x].iat[0]).ljust(ljust_values[ljust_index]) + "|" 
        ljust_index += 1
    return input_parameters_log_string
    
    

In [18]:
def satisfice(start_date_YYYYMMDD,initial_balances_dict,scheduled_items_df,accounts_df,days_out):
    
    logger.info('enter satisfice()')
    #logger.info('start_date_YYYYMMDD:'+str(start_date_YYYYMMDD))
    #logger.info('initial_balances_dict:'+str(initial_balances_dict))
    #logger.info('days_out:'+str(days_out))

    account_name_to_type_map = get_account_name_to_type_map(accounts_df)

    start_date = datetime.datetime.strptime(start_date_YYYYMMDD,'%Y-%m-%d')
    all_days = [(start_date + datetime.timedelta(days=d)).strftime('%Y-%m-%d') for d in range(0,days_out)]
    end_date_YYMMDD = max(all_days)

    #create first row
    initial_row_names = ['Date']
    for key in initial_balances_dict.keys():
        initial_row_names += [key]
    initial_row_names += ['Memo']

    initial_row = [start_date]
    for val in initial_balances_dict.values():
        initial_row += [val]
    initial_row += ['']

    initial_row_df = pd.DataFrame(initial_row).T
    initial_row_df.columns = initial_row_names


    satisfice_input_parameters_log_string = "\n"+'INITIAL BALANCES: Start_Date: '+str(start_date_YYYYMMDD)+'  Days_Out: '+str(days_out)+' End_Date: '+str(end_date_YYMMDD)+'\n'
    ljust_values=[]
    for x in initial_row_names[1:len(initial_row_names)-1]:
        x = x.replace("Interest","Intrst")
        x = x.replace("Balance","Bal")
        x = x.replace("Checking","Chk")
        x = x.replace("Savings","Sav")
        x = x.replace("Grace Period","GP")
        x = x.replace("Credit","Crdt")
        x = x.replace("Loan","Ln")
        ljust_values += [max(len(x),8)]
        satisfice_input_parameters_log_string += str(x).ljust(max(len(x),8)) + "|"

    satisfice_input_parameters_log_string += "\n"
    ljust_index = 0
    for bal_key in initial_balances_dict.keys():
        satisfice_input_parameters_log_string += str(initial_balances_dict[bal_key]).ljust(ljust_values[ljust_index]) + "|" 
        ljust_index += 1
    #logger.info(satisfice_input_parameters_log_string)


    loan_acct_names = []
    for acct_name in accounts_df['Account Name']:
        if 'Loan' in acct_name and 'Interest' not in acct_name:
            loan_acct_names += [acct_name]

    loan_account_name_to_interest_rate_map = {}
    for acct_name in loan_acct_names:
        loan_account_name_to_interest_rate_map[acct_name] = accounts_df[accounts_df['Account Name'] == acct_name]['Interest Rate APR'].iat[0]

    #credit_acct_name_billing_dt_tuples_list = []
    billing_dt_to_credit_acct_name_list_map = {}
    for index, row in accounts_df.iterrows():
        if 'Credit' in row['Account Name'] and 'Grace Period Balance' not in row['Account Name']:
            cc_start_date_YYYYMMDD = row['Billing Start Dt']
            cc_end_date_YYYYMMDD = (datetime.datetime.strptime(start_date_YYYYMMDD,'%Y-%m-%d') + datetime.timedelta(days=days_out)).strftime('%Y-%m-%d')

            #logger.info('cc_start_date_YYYYMMDD:'+str(cc_start_date_YYYYMMDD))
            #logger.info('cc_end_date_YYYYMMDD:'+str(cc_end_date_YYYYMMDD))
            for dt in generate_date_sequence( cc_start_date_YYYYMMDD, cc_end_date_YYYYMMDD,'monthly'):
                #credit_acct_name_billing_dt_tuples_list += [(row['Account Name'],dt)]

                #logger.info(str(min(all_days))+' ?<= '+str(dt)+' ?<= '+str(max(all_days))+' : '+str(min(all_days) <= dt.strftime('%Y-%m-%d') and dt.strftime('%Y-%m-%d') <= max(all_days)))
                if min(all_days) <= dt.strftime('%Y-%m-%d') and dt.strftime('%Y-%m-%d') <= max(all_days):
                    pass
                else:
                    continue

                if dt.strftime('%Y-%m-%d') in billing_dt_to_credit_acct_name_list_map.keys():
                    billing_dt_to_credit_acct_name_list_map[dt.strftime('%Y-%m-%d') ] += [row['Account Name']]
                else:
                    billing_dt_to_credit_acct_name_list_map[dt.strftime('%Y-%m-%d') ] = [row['Account Name']]

    previous_item_dt = initial_row_df['Date'].iat[0]                
    for d in all_days:
        #logger.info('start_date:'+str(start_date))
        #logger.info('d ?= start_date_YYYYMMDD:'+str(d == start_date_YYYYMMDD))
        #logger.info('d:'+str(d))
        if d == start_date_YYYYMMDD:
            previous_rows_df = initial_row_df
            continue            

        new_row_df = previous_rows_df.tail(1).copy()
        new_row_df.loc[0,"Date"] = d
        new_row_df.loc[0,"Memo"] = ""

        #logger.info('billing_dt_to_credit_acct_name_list_map.keys():'+str(billing_dt_to_credit_acct_name_list_map.keys()))
        if d in billing_dt_to_credit_acct_name_list_map.keys():

            #credit txns are first put in Grace Period Balance.
            #then, the first time through this loop, any balance gets moved to "Credit"
            #then, the second time though this loop, any balance in Credit will accrue interest

            grace_period_balance = new_row_df.loc[0,billing_dt_to_credit_acct_name_list_map[d][0]+' Grace Period Balance']
            interest_accruing_balance = new_row_df.loc[0,billing_dt_to_credit_acct_name_list_map[d][0]]

            accrued_interest = interest_accruing_balance * 0.2374/12 #TODO update this if i ever have multiple credit cards
            new_interest_accruing_balance = grace_period_balance + interest_accruing_balance + accrued_interest

            #todo: not totally sure this is right
            new_row_df.loc[0,billing_dt_to_credit_acct_name_list_map[d][0]+' Grace Period Balance'] = 0
            new_row_df.loc[0,billing_dt_to_credit_acct_name_list_map[d][0]] = new_interest_accruing_balance

            #logger.info('GRACE PERIOD LOGIC')
            #logger.info('Grace Period Balance.........:'+str(grace_period_balance))
            #logger.info('Interest Accruing Balance....:'+str(interest_accruing_balance))
            #logger.info('Accrued Interest.............:'+str(accrued_interest))
            #logger.info('New Interest Accruing Balance:'+str(new_interest_accruing_balance))



        #do daily interest accruals for this day
        for acct_name in loan_acct_names:
            old_interest_value = previous_rows_df.tail(1)[acct_name+" Interest"]
            old_principal_value = previous_rows_df.tail(1)[acct_name]

            new_marginal_interest = (old_principal_value*(loan_account_name_to_interest_rate_map[acct_name]/365.25)).iat[0]
            #logger.info('new_marginal_interest:'+str(new_marginal_interest))
            new_row_df.iloc[:,new_row_df.columns == (acct_name+' Interest')] += new_marginal_interest


        relevant_items_df = scheduled_items_df[ (scheduled_items_df['Date'] == d) & (scheduled_items_df['Priority'] == 1)]
        
        
        logger.debug('relevant_items_df:')
        logger.debug(relevant_items_df.to_string())
        for index, row in relevant_items_df.iterrows():
            assert sum(new_row_df.columns == row["Account From"]) == 1 or pd.isna(row["Account From"])
            assert sum(new_row_df.columns == row["Account To"]) == 1 or pd.isna(row["Account To"])
            
            #logger.info(str(previous_item_dt)+ ' ?= '+str(row["Date"])+' '+str(previous_item_dt == row["Date"]))

            row.Amount = float(row.Amount)
            #logger.info('row.Amount:'+str(row.Amount))

            if not pd.isna(row["Account To"]):
                account_to__type = account_name_to_type_map[row["Account To"]]
            else:
                account_to__type = ""

            if not pd.isna(row["Account From"]):
                account_from__type = account_name_to_type_map[row["Account From"]]
            else:
                account_from__type = ""

            #Loan Payments, from any account
            if account_to__type == 'Loan':
                old_principal_value = float(new_row_df[row["Account To"]].iat[0])
                old_interest_value = float(new_row_df[row["Account To"]+' Interest'].iat[0])

                #payment goes to interest only
                #logger.info('row.Amount*-1:'+str(row.Amount*-1)+' ?= '+str(old_interest_value))
                if row.Amount <= old_interest_value:
                    #principal remains unchanged
                    new_principal_value = old_principal_value
                    new_row_df[row["Account To"]]  = new_principal_value

                    new_interest_value = old_interest_value - row.Amount
                    new_row_df[row["Account To"]+' Interest'] = new_interest_value

                #this includes the case where interest = 0
                elif row.Amount > old_interest_value:

                    new_principal_value = old_principal_value + old_interest_value - row.Amount
                    new_row_df[row["Account To"]] = new_principal_value

                    #Interest is now 0
                    new_interest_value = 0
                    new_row_df[row["Account To"]+' Interest'] = new_interest_value

                else:
                    logger.error('error in satisfice::loan interest payment calculation')

                new_row_df[row["Account From"]] = new_row_df[row["Account From"]] - row.Amount

            elif account_from__type == 'Credit':
                #logger.debug('Case: FROM CREDIT')
                old_value = float(new_row_df[row["Account From"] + ' Grace Period Balance'].iat[0])
                new_value = old_value + float(row.Amount)

                #logger.info(str(d)+' '+str(index)+' '+str(row.Memo)+' ; '+str(row["Account From"] + ' Grace Period Balance')+' '+str(float(old_value))+' ('+str(float(row.Amount))+') -> '+str(float(new_value)))

                new_row_df[row["Account From"] + ' Grace Period Balance'] = new_value

                if account_to__type != '':
                    pass # i dont think this ever happens
                    raise NotImplementedError

            elif account_from__type == 'Checking' and account_to__type == 'Credit':
                old_from_value = float(new_row_df[row["Account From"]].iat[0])
                new_from_value = old_from_value - float(row.Amount)

                #could put a log statement here

                new_row_df.loc[:,new_row_df.columns == (row["Account From"])] = new_from_value

                #implement credit card payment
                old_grace_period_balance_value = new_row_df[row["Account To"] + ' Grace Period Balance'].iat[0]
                old_interest_accruing_balance_value = new_row_df[row["Account To"]].iat[0]

                total_cc_debt = ( old_grace_period_balance_value + old_interest_accruing_balance_value )

                if total_cc_debt == 0:
                    continue

                #logger.info('row.Amount:'+str(row.Amount))
                #logger.info('total_cc_debt:'+str(total_cc_debt))
                if row.Amount > total_cc_debt and total_cc_debt > 0:
                    logger.error("Attempting to make a credit card payment in excess of total debt. Exiting.")
                    raise ValueError

                #payment is applied to this-cycle balance first.
                if row.Amount <= old_grace_period_balance_value:
                    new_grace_period_balance_value = old_grace_period_balance_value - row.Amount
                    new_interest_accruing_balance_value = old_interest_accruing_balance_value

                #this includes where old_grace_period_balance_value is 0
                elif row.Amount > old_grace_period_balance_value:
                    new_grace_period_balance_value = 0
                    new_interest_accruing_balance_value = old_interest_accruing_balance_value + old_grace_period_balance_value - row.Amount

                else:
                    logger.error("Undefined case in satisfice while calculating credit card payment allocation.")
                    raise ValueError

                new_row_df[row["Account To"] + ' Grace Period Balance'] = new_grace_period_balance_value
                new_row_df[row["Account To"]] = new_interest_accruing_balance_value

                #logger.info('old_grace_period_balance_value:'+str(old_grace_period_balance_value))
                #logger.info('old_interest_accruing_balance_value:'+str(old_interest_accruing_balance_value))

                #logger.info('new_grace_period_balance_value:'+str(new_grace_period_balance_value))
                #logger.info('new_interest_accruing_balance_value:'+str(new_interest_accruing_balance_value))

                assert new_grace_period_balance_value >= 0
                assert new_interest_accruing_balance_value >= 0

            #Income
            elif account_from__type == '' and account_to__type == 'Checking':
                new_row_df[row["Account To"]] = new_row_df[row["Account To"]] + row.Amount

            else:
                #logger.info("row:")
                #logger.info(row.to_string())
                old_from_value = float(new_row_df[row["Account From"]].iat[0])
                new_from_value = old_from_value - float(row.Amount)

                #could put a log statement here

                new_row_df[row["Account From"]] = new_from_value

            new_row_df["Memo"] += str(row.Memo) + ' ; '
            previous_item_dt = row["Date"]
            #previous_rows_df = pd.concat([previous_rows_df,new_row_df])
        previous_rows_df = pd.concat([previous_rows_df,new_row_df])
    #pd.concat([previous_rows_df,new_row_df])


    satisficed_rows_df = previous_rows_df
    logger.info('exit satisfice()')
    return satisficed_rows_df


In [19]:
def main():
    parameter_tables_df = read_budget_item_excel_sheet('/home/hdickie/sandbox/Budget_Items.xlsx')
    
    
    parameter_tables_df[0]['Start Date'] = parameter_tables_df[0]['Start Date'].astype('string')
    
    accounts_df = parameter_tables_df[3]

    days_out = 90

    start_date_YYYYMMDD = datetime.datetime.now().strftime('%Y-%m-%d')
    scheduled_items_df = generate_series_from_item_schedule(parameter_tables_df[0],accounts_df
                                                            ,start_date_YYYYMMDD,days_out)

    logger.debug('scheduled_items_df:')
    logger.debug(scheduled_items_df.to_string())
    
    initial_balances_dict = get_initial_balances(accounts_df)

    current_ts_df = satisfice(start_date_YYYYMMDD,initial_balances_dict,scheduled_items_df,accounts_df,days_out)

    logger.debug('current_ts_df:')
    logger.debug(current_ts_df.to_string())
    
    
    output_df = optimize(current_ts_df,scheduled_items_df,accounts_df)
    #output_df = optimize_one_step(current_ts_df,proposed_transaction_df,accounts_df)
    #output_df = current_ts_df
    
    write_results(output_df)
    
    return output_df

In [34]:
def optimize(current_ts_df,scheduled_items_df,accounts_df):
    logger.info('enter optimize()')
    
    updated_current_ts_df = current_ts_df
    scheduled_items_priority_gt_2 = scheduled_items_df[scheduled_items_df['Priority'] > 1]
    
    logger.info('Priority    Count')
    p_counts_table = scheduled_items_priority_gt_2['Priority'].value_counts()
    for i in range(0,len(p_counts_table.index)):
        logger.info(str(p_counts_table.index[i]).ljust(11)+' '+str(p_counts_table.iloc[i]))
        
    
    relevant_priority_indicies = scheduled_items_priority_gt_2['Priority'].unique()
    for p in relevant_priority_indicies:
        
        current_priority_level_items = scheduled_items_priority_gt_2[scheduled_items_priority_gt_2['Priority'] == p]
        
        for index, row in current_priority_level_items.iterrows():
            #print(index)
            #print(row)
            
            if row["Date"] > max(updated_current_ts_df["Date"]):
                continue
            
            new_today_df, executed_or_deferred_or_skipped_transaction_df = optimize_one_step(updated_current_ts_df,row,accounts_df)
            #satisfice(start_date_YYYYMMDD,initial_balances_dict,scheduled_items_df,accounts_df,days_out)
            
            
            #logger.info('new_today_df:')
            #logger.info(new_today_df)
            
            initial_balances_dict = {}
            for cname in new_today_df.columns:
                if cname == "Date" or cname == "Memo":
                    continue
                initial_balances_dict[cname] = new_today_df[cname].iat[0]
            
            #logger.info('initial_balances_dict:'+str(initial_balances_dict))
            
            #logger.info('max(current_ts_df[Date]):'+str(max(current_ts_df['Date'])))
            #logger.info('new_today_df[Date]:'+str(new_today_df['Date']))
            new_days_out = ( max(current_ts_df['Date']) - new_today_df['Date'].iat[0] ).days
            start_date_YYYYMMDD = (new_today_df['Date'].iat[0] + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
            #logger.info('start_date_YYYYMMDD:'+str(start_date_YYYYMMDD))
            new_satisficed_rows = satisfice(start_date_YYYYMMDD,initial_balances_dict,scheduled_items_df,accounts_df,new_days_out)
            
            behind_rows_df = current_ts_df[current_ts_df["Date"] < start_date_YYYYMMDD ]
            ahead_rows_df = new_satisficed_rows
            logger.info('start_date_YYYYMMDD:'+str(start_date_YYYYMMDD))
            logger.info('row:'+str(row))
            logger.info('behind_rows_df:'+str(behind_rows_df.shape)+'  ahead_rows_df:'+str(ahead_rows_df.shape))
            updated_current_ts_df = pd.concat([behind_rows_df,ahead_rows_df])
            
            #assert min(updated_current_ts_df['Checking']) > 0
            
            current_ts_df = updated_current_ts_df
            #current_ts_df before item dt sats the same, item_dt is new_today_df, the remaining rows should be satisficed
            
            #Todo, defer items by increasing dt
        
             
    logger.info('exit optimize()')         
    return (current_ts_df)

def optimize_one_step(current_ts_df,proposed_transaction_df,accounts_df):
    logger.info('enter optimize_one_step()')
    
    date_of_proposed_transaction = proposed_transaction_df['Date']
    
    logger.info('Proposed Transaction:')
    logger.info('  Priority  Date        Memo')
    logger.info('  '+str(proposed_transaction_df['Priority']).ljust(10)+str(date_of_proposed_transaction.strftime('%Y-%m-%d'))+'  '+str(proposed_transaction_df['Memo']))
    
    
    today_df = current_ts_df[current_ts_df['Date'] == date_of_proposed_transaction]
    new_today_df = today_df.copy()
    today_and_the_future_df = current_ts_df[current_ts_df['Date'] >= date_of_proposed_transaction]
    
    
    if proposed_transaction_df['Date'] > max(current_ts_df['Date']):
        logger.info('Proposed transaction on or after the last date of time series, so we skip.')
        executed_or_deferred_or_skipped_transaction_df = proposed_transaction_df.head(0)
        return (new_today_df, executed_or_deferred_or_skipped_transaction_df)
    
    #logger.info('proposed_transaction_df:'+proposed_transaction_df.to_string())
    
    account_name_to_type_map = get_account_name_to_type_map(accounts_df)
    
    

    checking_lookahead = min(today_and_the_future_df['Checking'])
    free_balance = checking_lookahead

    if proposed_transaction_df.Amount == 'FREE_BALANCE' and account_name_to_type_map[proposed_transaction_df["Account To"]] == "Credit" :
        logger.info('Free Balance: '+str(free_balance))
        
        if checking_lookahead > 0 :
            #logger.info('Executing FREE_BALANCE transaction')
            
            #logger.info("Before Txn:")
            #logger.info(reformat_acct_balance_row_to_log_string(today_df))
            
            previous_Account_To_bal = today_df[proposed_transaction_df["Account To"]].iat[0]
            previous_Account_To_GP_bal = today_df[proposed_transaction_df["Account To"]+" Grace Period Balance"].iat[0]
            total_cc_debt = ( previous_Account_To_bal + previous_Account_To_GP_bal )
            
            #logger.info('previous_Account_To_bal:'+str(previous_Account_To_bal))
            #logger.info('previous_Account_To_GP_bal:'+str(previous_Account_To_GP_bal))
            if total_cc_debt == 0:
                logger.info('No CC debt, so we skip this additional cc payment')
                executed_or_deferred_or_skipped_transaction_df = proposed_transaction_df.head(0)
            else:
                
                #TODO
                #aaply first to grace period balance
   


                    
                #payment is applied to this-cycle balance first.
                new_free_balance = -1
                if free_balance <= previous_Account_To_GP_bal:
                    #logger.info('Free Balance <= Grace Period Balance')
                    new_grace_period_balance_value = previous_Account_To_GP_bal - free_balance
                    new_account_to_bal = previous_Account_To_bal
                    new_free_balance = 0
                #this includes where old_grace_period_balance_value is 0 and the free_balance is less than the total
                elif free_balance > previous_Account_To_GP_bal and ( free_balance <= previous_Account_To_bal + previous_Account_To_GP_bal ) :
                    #logger.info('Total CC Debt >= Free Balance > Grace Period Balance')
                    new_grace_period_balance_value = 0
                    new_account_to_bal = previous_Account_To_bal + previous_Account_To_GP_bal - free_balance
                    new_free_balance = 0
                elif free_balance >= ( previous_Account_To_bal + previous_Account_To_GP_bal ):
                    #logger.debug('Free Balance > Total CC Debt')
                    new_grace_period_balance_value = 0
                    new_account_to_bal = 0
                    new_free_balance = ( previous_Account_To_bal + previous_Account_To_GP_bal ) - free_balance
                else:
                    logger.error("Undefined case in optimize_one_step while calculating credit card payment allocation.")
                    raise ValueError
                    
                new_today_df.loc[:,today_df.columns == (proposed_transaction_df["Account To"] + ' Grace Period Balance')] = new_grace_period_balance_value
                new_today_df.loc[:,today_df.columns == (proposed_transaction_df["Account To"])] = new_account_to_bal
                new_today_df.loc[:,today_df.columns == (proposed_transaction_df["Account From"])] = new_free_balance
                new_today_df.loc[:,today_df.columns == (proposed_transaction_df["Memo"])] += ' ; ' + str(proposed_transaction_df["Memo"])
                
                executed_or_deferred_or_skipped_transaction_df = proposed_transaction_df.head(0)
    
    
    #satisfice with new today row
    #logger.info("After Txn:")
    #logger.info(reformat_acct_balance_row_to_log_string(new_today_df))
    
    logger.info('new_today_df:')
    logger.info(new_today_df.to_string())
    logger.info('exit optimize_one_step()')
    return (new_today_df, executed_or_deferred_or_skipped_transaction_df)

In [35]:
initialize_logging(log_level="info")
output_df = main()
#validate_output(output_df)

2022-06-06 21:04:03,319 - Expense_Forecast - INFO - enter satisfice()
2022-06-06 21:04:03,861 - Expense_Forecast - INFO - exit satisfice()
2022-06-06 21:04:03,868 - Expense_Forecast - INFO - enter optimize()
2022-06-06 21:04:03,869 - Expense_Forecast - INFO - Priority    Count
2022-06-06 21:04:03,871 - Expense_Forecast - INFO - 2           4
2022-06-06 21:04:03,873 - Expense_Forecast - INFO - enter optimize_one_step()
2022-06-06 21:04:03,874 - Expense_Forecast - INFO - Proposed Transaction:
2022-06-06 21:04:03,875 - Expense_Forecast - INFO -   Priority  Date        Memo
2022-06-06 21:04:03,876 - Expense_Forecast - INFO -   2         2022-06-06  Pay cc balance in excess of minimum
2022-06-06 21:04:03,880 - Expense_Forecast - INFO - Free Balance: 258.48
2022-06-06 21:04:03,882 - Expense_Forecast - INFO - new_today_df:
2022-06-06 21:04:03,886 - Expense_Forecast - INFO -         Date  Checking Savings  Credit 1  Credit 1 Grace Period Balance   Loan A Loan A Interest   Loan B Loan B Interes

In [13]:
output_df

,Date,Checking,Savings,Credit 1,Credit 1 Grace Period Balance,Loan A,Loan A Interest,Loan B,Loan B Interest,Loan C,Loan C Interest,Loan D,Loan D Interest,Loan E,Loan E Interest,Memo
0,2022-06-06,298.48,50.0,2434.88,1832.54,4746.18,0.0,1919.55,0.0,4726.68,0.0,1823.31,0.0,3359.17,0.0,
0,2022-06-07,298.48,50.0,2434.88,1832.54,4746.18,0.0,1919.55,0.0,4726.68,0.0,1823.31,0.0,3359.17,0.0,
0,2022-06-08,298.48,50.0,2434.88,1862.54,4746.18,0.557457,1919.55,0.225458,4726.68,0.48658,1823.31,0.187697,3359.17,0.428576,Food ;
0,2022-06-09,298.48,50.0,2434.88,1892.54,4746.18,1.114914,1919.55,0.450917,4726.68,0.973159,1823.31,0.375395,3359.17,0.857152,Food ;
0,2022-06-10,298.48,50.0,2434.88,1922.54,4746.18,1.672371,1919.55,0.676375,4726.68,1.459739,1823.31,0.563092,3359.17,1.285727,Food ;
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,2022-08-30,6772.08,50.0,6495.482116,897.0,4688.565278,14.868625,1842.979937,5.844555,4665.032948,12.966301,1744.601866,4.849062,3294.186338,11.347694,Food ;
0,2022-08-31,6772.08,50.0,6495.482116,927.0,4688.565278,15.419314,1842.979937,6.06102,4665.032948,13.446534,1744.601866,5.028657,3294.186338,11.767979,Food ;
0,2022-09-01,4772.08,50.0,6495.482116,957.0,4688.565278,15.970004,1842.979937,6.277485,4665.032948,13.926768,1744.601866,5.208252,3294.186338,12.188264,Rent ; Food ;
0,2022-09-02,4772.08,50.0,6495.482116,987.0,4688.565278,16.520694,1842.979937,6.49395,4665.032948,14.407001,1744.601866,5.387846,3294.186338,12.608549,Food ;
